# Week 12 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 데이터와 자동 확인 도우미

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

CAFE = pd.read_csv(get_data("cafe_sales.csv"))           # date는 아직 문자열
WEATHER = pd.read_csv(get_data("weather_daily.csv"))
CAFE_BACKUP = CAFE.copy()
WEEKDAYS = ["월", "화", "수", "목", "금", "토", "일"]
DAYPARTS = ["아침", "점심", "오후", "저녁"]


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 시간 파생 변수 만들기 (2점)

`add_features(cafe)`는 원본을 바꾸지 않고 다음 열을 추가(또는 변환)한 새 DataFrame을 돌려줍니다.

| 열 | 내용 |
|---|---|
| `date` | 문자열 → `datetime` |
| `revenue` | `unit_price × qty` |
| `month` | 월 (정수) |
| `weekday` | `"월"`~`"일"` 한 글자 |
| `hour` | `time`의 시 (정수) |
| `daypart` | 8~10시 `"아침"`, 11~13시 `"점심"`, 14~16시 `"오후"`, 17~19시 `"저녁"` (문자열) |

In [2]:
def add_features(cafe):
    out = cafe.copy()
    out["date"] = pd.to_datetime(out["date"])
    out["revenue"] = out["unit_price"] * out["qty"]
    out["month"] = out["date"].dt.month
    out["weekday"] = out["date"].dt.dayofweek.map(dict(enumerate(WEEKDAYS)))
    out["hour"] = out["time"].str[:2].astype(int)
    out["daypart"] = pd.cut(out["hour"], bins=[7, 10, 13, 16, 19], labels=DAYPARTS).astype(str)
    return out

In [3]:
check("새 열", lambda: {"revenue", "month", "weekday", "hour", "daypart"} <= set(add_features(CAFE).columns))
check("date는 datetime", lambda: add_features(CAFE)["date"].dtype.kind == "M")
check("매출 합계", lambda: add_features(CAFE)["revenue"].sum() == 14_198_000)
check("요일 값", lambda: set(add_features(CAFE)["weekday"]) == set("월화수목금토"))
check("시간대 경계 (10시=아침, 11시=점심, 19시=저녁)", lambda: add_features(CAFE).groupby("hour")["daypart"].first()[[10, 11, 16, 17, 19]].tolist()
      == ["아침", "점심", "오후", "저녁", "저녁"])
check("원본 불변", lambda: (add_features(CAFE), CAFE.equals(CAFE_BACKUP))[1])

✅ 통과 - 새 열
✅ 통과 - date는 datetime
✅ 통과 - 매출 합계
✅ 통과 - 요일 값
✅ 통과 - 시간대 경계 (10시=아침, 11시=점심, 19시=저녁)
✅ 통과 - 원본 불변


---
## 문제 2. 요일 × 시간대 일평균 매출표 (2점)

`weekday_daypart(cafe)`는 **요일(행, 월~토 순서) × 시간대(열, 아침·점심·오후·저녁 순서)** 의 **일평균 매출(원, 정수 반올림)** 표를 돌려줍니다.

- 일평균 = 그 칸의 매출 합계 ÷ **그 요일의 영업일 수** (데이터에 그 요일이 나온 날짜 수)
- 문제 1의 `add_features`를 재사용하세요.

In [4]:
def weekday_daypart(cafe):
    f = add_features(cafe)
    days = f.groupby("weekday")["date"].nunique()
    table = f.pivot_table(index="weekday", columns="daypart", values="revenue", aggfunc="sum", fill_value=0)
    table = table.div(days, axis=0).round(0).astype(int)
    order = [d for d in WEEKDAYS if d in table.index]
    table = table.loc[order, DAYPARTS]
    table.index.name, table.columns.name = "weekday", "daypart"
    return table

In [5]:
check("모양과 순서", lambda: list(weekday_daypart(CAFE).index) == list("월화수목금토")
      and list(weekday_daypart(CAFE).columns) == DAYPARTS)
check("월요일 점심 일평균", lambda: weekday_daypart(CAFE).loc["월", "점심"] == 65_888)
check("토요일 저녁 일평균", lambda: weekday_daypart(CAFE).loc["토", "저녁"] == 8_473)
check("가장 바쁜 칸은 화요일 점심", lambda: weekday_daypart(CAFE).stack().idxmax() == ("화", "점심"))

✅ 통과 - 모양과 순서
✅ 통과 - 월요일 점심 일평균
✅ 통과 - 토요일 저녁 일평균
✅ 통과 - 가장 바쁜 칸은 화요일 점심


---
## 문제 3. 메뉴 매출 비중과 상위 80% 메뉴 (3점)

1. `menu_share(cafe)`: 메뉴를 인덱스로, **매출 내림차순** 정렬. 열은
   - `qty`(판매 수량), `revenue`(매출), `share`(매출 비중 %, 소수 첫째 자리), `cum_share`(누적 비중 %, 소수 첫째 자리)
   - `cum_share`는 반올림 **전** 비중을 누적한 뒤 반올림합니다.
2. `top80(cafe)`: 누적 비중이 **처음으로 80% 이상이 되는 메뉴까지**의 메뉴 이름 리스트 (매출 순)

In [6]:
def menu_share(cafe):
    f = add_features(cafe)
    m = f.groupby("menu").agg(qty=("qty", "sum"), revenue=("revenue", "sum")).sort_values("revenue", ascending=False)
    raw_share = m["revenue"] / m["revenue"].sum() * 100
    m["share"] = raw_share.round(1)
    m["cum_share"] = raw_share.cumsum().round(1)
    return m


def top80(cafe):
    m = menu_share(cafe)
    raw_cum = m["revenue"].cumsum() / m["revenue"].sum() * 100
    n = int((raw_cum < 80).sum()) + 1
    return list(m.index[:n])

In [7]:
check("열 구성과 정렬", lambda: list(menu_share(CAFE).columns) == ["qty", "revenue", "share", "cum_share"]
      and menu_share(CAFE)["revenue"].is_monotonic_decreasing)
check("1위 메뉴와 비중", lambda: (menu_share(CAFE).index[0], menu_share(CAFE)["share"].iloc[0]) == ("아메리카노", 23.6))
check("누적 비중 마지막 100", lambda: menu_share(CAFE)["cum_share"].iloc[-1] == 100.0)
check("top80", lambda: top80(CAFE) == ["아메리카노", "카페라떼", "바닐라라떼", "콜드브루", "레몬에이드", "녹차라떼", "베이글"])
tiny = pd.DataFrame({"date": ["2025-03-03"] * 3, "time": ["09:00"] * 3, "menu": ["A", "B", "C"],
                     "unit_price": [800, 100, 100], "qty": [1, 1, 1]})
check("정확히 80%면 거기서 멈춤", lambda: top80(tiny) == ["A"])

✅ 통과 - 열 구성과 정렬
✅ 통과 - 1위 메뉴와 비중
✅ 통과 - 누적 비중 마지막 100
✅ 통과 - top80
✅ 통과 - 정확히 80%면 거기서 멈춤


---
## 문제 4. 기온 구간 분석 (AI 협업, 3점)

`temp_bands(cafe, weather)`는 영업일마다 매출·아이스 비율을 구하고 기온과 합쳐, **기온 구간별 요약표**를 돌려줍니다.

1. 일별 `revenue`(매출 합계), `ice_ratio` = ICE 판매 수량 ÷ (ICE + HOT 판매 수량) (`temp_type`이 `"-"`인 디저트는 제외)
2. 날씨의 `avg_temp`를 날짜로 합칩니다. **합친 뒤 영업일 수가 그대로인지** 확인하고, 다르면 `ValueError`
3. 구간: `10℃ 미만`, `10~15`, `15~20`, `20~25`, `25℃ 이상` (`pd.cut`, 경계: −∞, 10, 15, 20, 25, ∞ / 오른쪽 경계 포함 기본값)
4. 결과 열: `days`(일수), `revenue`(일평균 매출, 정수 반올림), `ice_pct`(평균 아이스 비율 %, 소수 첫째 자리)

**AI 협업 과제**: AI에게 이 분석을 요청하면서 **"상관을 인과로 해석하지 말라"**, **"일평균으로 비교하라"** 를 프롬프트에 넣었는지 기록하세요. 그리고 AI가 쓴 해석 문장 하나를 결과표로 검증해 아래 기록에 적습니다.

In [8]:
def temp_bands(cafe, weather):
    f = add_features(cafe)
    w = weather.copy()
    w["date"] = pd.to_datetime(w["date"])
    daily = f.groupby("date").agg(revenue=("revenue", "sum")).reset_index()
    drinks = f[f["temp_type"].isin(["HOT", "ICE"])]
    ice = drinks.pivot_table(index="date", columns="temp_type", values="qty", aggfunc="sum", fill_value=0)
    ice["ice_ratio"] = ice["ICE"] / (ice["ICE"] + ice["HOT"])
    day = daily.merge(ice[["ice_ratio"]].reset_index(), on="date", how="left")
    merged = day.merge(w[["date", "avg_temp"]], on="date", how="left")
    if len(merged) != len(daily):
        raise ValueError("날짜 병합 후 영업일 수가 바뀌었습니다")
    band = pd.cut(merged["avg_temp"], bins=[-np.inf, 10, 15, 20, 25, np.inf],
                  labels=["10℃ 미만", "10~15", "15~20", "20~25", "25℃ 이상"])
    out = merged.groupby(band, observed=True).agg(days=("date", "size"), revenue=("revenue", "mean"),
                                                  ice_pct=("ice_ratio", "mean"))
    out["revenue"] = out["revenue"].round(0).astype(int)
    out["ice_pct"] = (out["ice_pct"] * 100).round(1)
    out.index = out.index.astype(str)
    return out

In [9]:
check("구간 5개 순서", lambda: list(temp_bands(CAFE, WEATHER).index) == ["10℃ 미만", "10~15", "15~20", "20~25", "25℃ 이상"])
check("일수 합계 = 영업일 95", lambda: temp_bands(CAFE, WEATHER)["days"].sum() == 95)
check("아이스 비율이 구간마다 증가", lambda: temp_bands(CAFE, WEATHER)["ice_pct"].is_monotonic_increasing)
check("25℃ 이상 행", lambda: temp_bands(CAFE, WEATHER).loc["25℃ 이상"].tolist() == [6, 168017, 81.3])
dup_weather = pd.concat([WEATHER, WEATHER.iloc[[70]]])                 # 날짜가 중복된 날씨 표


def _raises(exc, fn, *args):
    try:
        fn(*args)
    except exc:
        return True
    return False

check("병합으로 행이 늘면 ValueError", lambda: _raises(ValueError, temp_bands, CAFE, dup_weather))

✅ 통과 - 구간 5개 순서


✅ 통과 - 일수 합계 = 영업일 95
✅ 통과 - 아이스 비율이 구간마다 증가


✅ 통과 - 25℃ 이상 행
✅ 통과 - 병합으로 행이 늘면 ValueError


### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) Gemini |
| 해석 조건 | "상관을 인과로 해석하지 말 것, 일평균 비교, 각 주장마다 근거 표 제시" |
| AI 문장 | "기온이 25℃를 넘으면 매출이 크게 늘어 여름 성수기 전략이 필요합니다." |
| 검증 | 25℃ 이상은 6일뿐이고 일평균 매출 차이도 다른 구간과 비슷한 수준(14~17만 원) → **과장**. 뚜렷한 것은 아이스 비율(81.3%)이다 |

---
## 최종 프로젝트 제안서 (문제 4에 포함)

다음 형식으로 작성합니다. 13주차에 피드백을 받고 수정합니다. 자세한 기준은 [최종 프로젝트 안내](../../projects/final-project.md)를 참고하세요.

**제안서 채점 포인트**

| 항목 | 확인 |
|---|---|
| 연구 질문 | 데이터로 답할 수 있고, 비교 대상이 있으며, 구체적인가 |
| 데이터 | 출처·이용 조건·한 행의 의미가 명확한가 |
| 방법 | 정제·집계·그래프 계획이 질문과 연결되는가 |
| 한계 | 표본 크기, 가상 데이터, 상관과 인과 등 한계를 알고 있는가 |
| AI 계획 | 검증 방법까지 구체적인가 |

---
## 성찰 (필수)

1. "한 행의 의미"를 먼저 확인한 것이 분석에 어떤 도움이 되었나요?
2. 합계 대신 일평균으로 바꾸었을 때 결론이 달라진 예가 있었나요?
3. AI의 해석 문장을 검증하면서 배운 점은?

(학생 작성 영역)